# 03. 게임 성과 등급 정의 및 분류

**분석 목적:** 전체 인디게임 데이터를 대상으로 Steam 리뷰 시스템 기준에 따라 성과 등급을 정의하고 파생 컬럼으로 생성한다.

**사용 데이터:** `data/preprocessed/steam_indie_games.csv` (9,692개, 리뷰 10개 이상, 2023~2025년, EA·F2P 제외)

**분석 흐름:**
1. 성과 등급 기준 정의
2. 데이터 로드
3. 성과 등급 파생 컬럼 생성
4. 등급별 분포 확인

## 1. 성과 등급 기준 정의

두 축을 모두 Steam 리뷰 시스템 기준으로 설정한다.

**축 기준 설정**

| 축 | 등급 | 값 | 기준 | 근거 |
|---|---|---|---|---|
| 규모 (리뷰 수) | 높음 | `high` | 500개 이상 | Steam 등급 상위 구간 |
| 규모 (리뷰 수) | 중간 | `mid` | 50개~499개 | Steam 등급 중간 구간 |
| 규모 (리뷰 수) | 낮음 | `low` | 10개~49개 | Steam 등급 하위 구간 |
| 만족도 (긍정률) | 높음 | `high` | 80% 이상 | Steam Very Positive 이상 |
| 만족도 (긍정률) | 중간 | `mid` | 70%~79% | Steam Mostly Positive |
| 만족도 (긍정률) | 낮음 | `low` | 70% 미만 | Steam Mixed 이하 |

**등급표 (`performance_grade` = `scale_grade` + `_` + `satisfaction_grade`)**

| 규모 \ 만족도 | 높음 (`high`) | 중간 (`mid`) | 낮음 (`low`) |
|---|---|---|---|
| **높음** (`high`) | `high_high` | `high_mid` | `high_low` |
| **중간** (`mid`) | `mid_high` | `mid_mid` | `mid_low` |
| **낮음** (`low`) | `low_high` | `low_mid` | `low_low` |

**대시보드 표시용 레이블 매핑**

| performance_grade | 한국어 레이블 |
|---|---|
| `high_high` | 대흥행 |
| `high_mid` | 상업적 성공 |
| `high_low` | 호불호 |
| `mid_high` | 숨겨진 명작 |
| `mid_mid` | 평범 |
| `mid_low` | 외면 |
| `low_high` | 니치 (틈새) |
| `low_mid` | 미노출 |
| `low_low` | 미반응 |

## 2. 라이브러리 로드 및 데이터 로드

In [6]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

games = pd.read_csv('../../data/preprocessed/steam_indie_games.csv')
games['positive_rate'] = games['positive'] / games['total_reviews'] * 100

print(f'전체 게임: {len(games):,}개')
games.head(3)

전체 게임: 8,730개


,appid,positive,negative,price,genres,total_reviews,name,developers,release_date,short_description,...,categories,windows,mac,linux,recommendations_total,achievements_total,owners_lower,owners_higher,tags,positive_rate
0,226620,1912,364,14.99,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",2276,Desktop Dungeons,QCF Design,2023-04-18,Each step into the unknown heals you and revea...,...,"Single-player, Steam Achievements, Steam Tradi...",True,True,True,1129.0,35.0,200000,500000,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ...",84.007030
1,251570,327889,42157,44.99,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",370046,7 Days to Die,The Fun Pimps,2024-07-25,7 Days to Die is an open-world game that is a ...,...,"Single-player, Multi-player, PvP, Online PvP, ...",True,True,True,271581.0,43.0,10000000,20000000,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""...",88.607633
2,252190,157,98,19.99,"['Indie', 'RPG', 'Strategy']",255,Defender's Quest 2: Mists of Ruin,"Level Up Labs, LLC",2025-01-30,Rise above the Mirk and protect your ship from...,...,"Single-player, Steam Achievements, Full contro...",True,False,False,236.0,33.0,0,20000,"{""2D"": 70, ""RPG"": 94, ""Indie"": 59, ""Sci-fi"": 4...",61.568627


## 3. 성과 등급 파생 컬럼 생성

In [7]:
def assign_scale_grade(total_reviews):
    if total_reviews >= 500:
        return 'high'
    elif total_reviews >= 50:
        return 'mid'
    else:
        return 'low'

def assign_satisfaction_grade(positive_rate):
    if positive_rate >= 80:
        return 'high'
    elif positive_rate >= 70:
        return 'mid'
    else:
        return 'low'

GRADE_LABEL = {
    'high_high': '대흥행',
    'high_mid' : '상업적 성공',
    'high_low' : '호불호',
    'mid_high' : '숨겨진 명작',
    'mid_mid'  : '평범',
    'mid_low'  : '외면',
    'low_high' : '니치 (틈새)',
    'low_mid'  : '미노출',
    'low_low'  : '미반응',
}

GRADE_ORDER = list(GRADE_LABEL.keys())

games['scale_grade']        = games['total_reviews'].apply(assign_scale_grade)
games['satisfaction_grade'] = games['positive_rate'].apply(assign_satisfaction_grade)
games['performance_grade']  = games['scale_grade'] + '_' + games['satisfaction_grade']
games['performance_grade']  = pd.Categorical(games['performance_grade'], categories=GRADE_ORDER, ordered=True)

print('파생 컬럼 생성 완료:')
print('  scale_grade       : 규모 등급 (high/mid/low)')
print('  satisfaction_grade: 만족도 등급 (high/mid/low)')
print('  performance_grade : 성과 등급 (high_high ~ low_low)')
print()
games[['name', 'total_reviews', 'positive_rate', 'scale_grade', 'satisfaction_grade', 'performance_grade']].head(10)

파생 컬럼 생성 완료:
  scale_grade       : 규모 등급 (high/mid/low)
  satisfaction_grade: 만족도 등급 (high/mid/low)
  performance_grade : 성과 등급 (high_high ~ low_low)



,name,total_reviews,positive_rate,scale_grade,satisfaction_grade,performance_grade
0,Desktop Dungeons,2276,84.007030,high,high,high_high
1,7 Days to Die,370046,88.607633,high,high,high_high
2,Defender's Quest 2: Mists of Ruin,255,61.568627,mid,low,mid_low
3,Secrets of Grindea,8270,89.334946,high,high,high_high
4,Dwelvers,300,64.333333,mid,low,mid_low
5,FaeVerse Alchemy,360,67.500000,mid,low,mid_low
6,Bulwark: Falconeer Chronicles,1149,84.682332,high,high,high_high
7,Skin Deep,929,95.371367,high,high,high_high
8,Tempopo,44,97.727273,low,high,low_high
9,SinaRun,177,77.401130,mid,mid,mid_mid


## 4. 등급별 분포 확인

In [8]:
grade_stats = (
    games.groupby('performance_grade', observed=True)
    .agg(
        게임수=('appid', 'count'),
        리뷰수_중앙값=('total_reviews', 'median'),
        긍정률_중앙값=('positive_rate', 'median'),
    )
    .round(2)
)
grade_stats['비율(%)'] = (grade_stats['게임수'] / len(games) * 100).round(1)
grade_stats['리뷰수_중앙값'] = grade_stats['리뷰수_중앙값'].astype(int)
grade_stats.index = [GRADE_LABEL[g] for g in grade_stats.index]

display(grade_stats)

,게임수,리뷰수_중앙값,긍정률_중앙값,비율(%)
대흥행,815,1602,91.34,9.3
상업적 성공,161,851,76.42,1.8
호불호,78,1063,63.49,0.9
숨겨진 명작,1941,117,91.55,22.2
평범,509,126,75.64,5.8
외면,386,103,62.71,4.4
니치 (틈새),3490,19,94.12,40.0
미노출,552,20,75.00,6.3
미반응,798,19,58.33,9.1


In [9]:
grade_counts = (
    games['performance_grade'].value_counts()
    .reindex(GRADE_ORDER)
    .reset_index()
)
grade_counts.columns = ['grade', 'count']
grade_counts['label'] = grade_counts['grade'].map(GRADE_LABEL)

GRADE_COLOR = {
    'high_high': '#2d6a4f',
    'high_mid' : '#52b788',
    'high_low' : '#b7e4c7',
    'mid_high' : '#4C72B0',
    'mid_mid'  : '#adb5bd',
    'mid_low'  : '#DD8452',
    'low_high' : '#8172B2',
    'low_mid'  : '#e07a5f',
    'low_low'  : '#C44E52',
}

fig = px.bar(
    grade_counts,
    x='label', y='count',
    color='grade',
    color_discrete_map=GRADE_COLOR,
    text='count',
    title='성과 등급별 게임 수',
    labels={'label': '성과 등급', 'count': '게임 수'},
    category_orders={'label': [GRADE_LABEL[g] for g in GRADE_ORDER]},
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

**해석:** 전체 인디게임의 약 절반 가까이가 부진 구간에 속하며, 흥행작은 약 9% 수준이다. Steam 인디게임 시장에서 리뷰 500개 이상 + 긍정률 80% 이상을 동시에 달성하는 것이 얼마나 어려운지를 보여준다.

## 5. 성과 등급 히트맵 (3×3)

In [13]:
scale_order  = ['high', 'mid', 'low']
sat_order    = ['high', 'mid', 'low']
scale_labels = ['높음 (500+)', '중간 (50~499)', '낮음 (10~49)']
sat_labels   = ['높음 (80%+)', '중간 (70~79%)', '낮음 (~70%)']

pivot = (
    games.groupby(['scale_grade', 'satisfaction_grade'])
    .size()
    .unstack()
    .reindex(index=scale_order, columns=sat_order)
)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=sat_labels,
    y=scale_labels,
    text=pivot.values,
    texttemplate='%{text:,}',
    colorscale='Blues',
    colorbar=dict(title='게임 수'),
))
fig.update_layout(
    title='성과 등급 분포 히트맵 (규모 × 만족도)',
    xaxis_title='만족도 (긍정률)',
    yaxis_title='규모 (리뷰 수)',
    width=600,
    height=420,
)
fig.show()

## 6. 성과 등급 산점도 (리뷰 수 × 긍정률, log scale)

In [11]:
fig = px.scatter(
    games,
    x='total_reviews',
    y='positive_rate',
    color='performance_grade',
    color_discrete_map=GRADE_COLOR,
    hover_data={'name': True, 'total_reviews': True, 'positive_rate': ':.1f', 'performance_grade': False},
    opacity=0.45,
    log_x=True,
    title='게임 분포 산점도 (리뷰 수 × 긍정률)',
    labels={
        'total_reviews': '리뷰 수 (log scale)',
        'positive_rate': '긍정률 (%)',
        'performance_grade': '성과 등급',
    },
)

for x_boundary in [50, 500]:
    fig.add_vline(x=x_boundary, line_dash='dash', line_color='gray', line_width=1, opacity=0.6)
for y_boundary in [70, 80]:
    fig.add_hline(y=y_boundary, line_dash='dash', line_color='gray', line_width=1, opacity=0.6)

fig.update_layout(
    legend_title_text='성과 등급',
    legend=dict(itemsizing='constant'),
    width=850,
    height=520,
)
fig.show()

## 7. 데이터 저장

In [12]:
out_path = '../../data/preprocessed/steam_indie_games_graded.csv'

games['grade_label'] = games['performance_grade'].map(GRADE_LABEL)
games['performance_grade'] = games['performance_grade'].astype(str)
games.to_csv(out_path, index=False)

print(f'저장 완료 → {out_path}')
print(f'shape: {games.shape}')
print(f'추가된 컬럼: scale_grade, satisfaction_grade, performance_grade, positive_rate, grade_label')

저장 완료 → ../../data/preprocessed/steam_indie_games_graded.csv
shape: (8730, 25)
추가된 컬럼: scale_grade, satisfaction_grade, performance_grade, positive_rate, grade_label
